# Relax Structure with FAIRChem UMA — Universal Machine-learning Force Field

Use FAIRChem's [UMA](https://github.com/FAIR-Chem/fairchem) (Universal Model for Atoms) to relax crystal structures using machine-learned interatomic potentials.

<h2 style="color:green">Usage</h2>

1. Drop the materials files into the "uploads" folder in the JupyterLab file browser
1. Set Input Parameters below or use the default values
1. Click "Run" > "Run All" to run all cells
1. Wait for the run to complete. Scroll down to view cell results.
1. Review the relaxation plot and modify parameters as needed

## Methodology

1. Load materials from JSON files and create structure via `mat3ra-made`
2. Install FAIRChem UMA and apply Pyodide patches
3. Convert to ASE atoms with `to_ase()`
4. Relax the structure with FAIRChem UMA and visualize convergence
5. Compute relaxation energy

## 1. Set Input Parameters
### 1.1. Structure and Relaxation Parameters


In [ ]:
FOLDER = "uploads"
STRUCTURE_NAME = "Interface"  # Name of the structure to load from local file

RELAXATION_PARAMETERS = {
    "FMAX": 0.05,
}
UMA_TASK = "s2ef"  # "s2ef" (structure to energy/forces) or "is2re" (initial structure to relaxed energy)

## 2. Install Packages

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples|torch|uma")

    from mat3ra.notebooks_utils.pyodide.packages.torch import apply_all_patches

    apply_all_patches(include_fairchem=True)

## 3. Load Materials

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.standata.materials import Materials

structure = load_material_from_folder(FOLDER, STRUCTURE_NAME) or Material.create(
    Materials.get_by_name_first_match(STRUCTURE_NAME))

### 3.1. Visualize Input Structure

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

visualize([{"material": structure, "title": structure.name}], viewer=ViewersEnum.wave)
visualize(structure, repetitions=[1, 1, 1], rotation="-90x")

## 4. Apply Relaxation
### 4.1. Relax with FAIRChem UMA

In [ ]:
import plotly.graph_objs as go
from IPython.display import display
from plotly.subplots import make_subplots

from mat3ra.made.tools.convert import to_ase
from ase.optimize import BFGS

from fairchem.core import FAIRChemCalculator, pretrained_mlip

# Load UMA model
predictor = pretrained_mlip.get_predict_unit("uma-s-1", device="cpu")
calculator = FAIRChemCalculator(predictor, task_name=UMA_TASK)

ase_structure = to_ase(structure)
ase_structure.set_calculator(calculator)
dyn = BFGS(ase_structure)

steps = []
energies = []

fig = make_subplots(rows=1, cols=1, specs=[[{"type": "scatter"}]])
scatter = go.Scatter(x=[], y=[], mode="lines+markers", name="Energy")
fig.add_trace(scatter)
fig.update_layout(title_text="Real-time Optimization Progress", xaxis_title="Step", yaxis_title="Energy (eV)")

f = go.FigureWidget(fig)
display(f)


def plotly_callback():
    step = dyn.nsteps
    energy = ase_structure.get_total_energy()
    steps.append(step)
    energies.append(energy)
    print(f"Step: {step}, Energy: {energy:.4f} eV")
    with f.batch_update():
        f.data[0].x = steps
        f.data[0].y = energies


dyn.attach(plotly_callback, interval=1)
dyn.run(fmax=RELAXATION_PARAMETERS["FMAX"])

ase_original_structure = to_ase(structure)
ase_original_structure.set_calculator(calculator)
ase_final_structure = ase_structure

original_energy = ase_original_structure.get_total_energy()
relaxed_energy = ase_structure.get_total_energy()

print(f"The final energy is {float(relaxed_energy):.3f} eV.")

## 5. Analyze Results
### 5.1. View Structure Before and After Relaxation

In [ ]:
from mat3ra.made.tools.convert import from_ase

material_original = Material.create(from_ase(ase_original_structure))
material_relaxed = Material.create(from_ase(ase_final_structure))
material_original.name = structure.name
material_relaxed.name = structure.name + " (UMA Relaxed)"

visualize(
    [
        {"material": material_original, "title": material_original.name},
        {"material": material_relaxed, "title": material_relaxed.name},
    ],
    viewer=ViewersEnum.wave,
)

### 5.2. Output interlayer distance before and after relaxation
This requires labels for substrate and film present in the interface structure, which is already done if interface was created with `mat3ra-made`.

In [ ]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance

SUBSTRATE_TAG = 0
FILM_TAG = 1

print(
    f"Interlayer distance before relaxation: {get_average_interlayer_distance(material_original, SUBSTRATE_TAG, FILM_TAG):.4f} Å")
print(
    f"Interlayer distance after relaxation:  {get_average_interlayer_distance(material_relaxed, SUBSTRATE_TAG, FILM_TAG):.4f} Å")

## References

[1] FAIRChem: https://github.com/FAIR-Chem/fairchem  
[2] UMA — Universal Machine-learning Force Field for Atomistic Systems: https://arxiv.org/abs/2410.22570  
[3] mat3ra-made interface builder: https://github.com/Exabyte-io/made  